# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


In [ ]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [ ]:
# Task 1 — Heatmap: content by rating and release decade

df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
filtered = df[df['rating'].isin(ratings_filter)]

pivot = filtered.pivot_table(
    index='rating', columns='decade', values='type',
    aggfunc='count', fill_value=0
)

fig = px.imshow(
    pivot,
    color_continuous_scale='Blues',
    text_auto=True,
    title='TV-MA & TV-14 Dominate Every Decade — PG-13 Movies Peaked in the 2000s'
)
fig.update_layout(
    xaxis_title='Decade',
    yaxis_title='Content Rating',
    coloraxis_colorbar_title='Titles'
)
fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [ ]:
# Task 2 — Waterfall: Movie library growth 2015–2022

# Self-contained reload to avoid kernel state issues
_df = pd.read_csv('../data/netflix_catalogue.csv')

movies = _df[(_df['type'] == 'Movie') & (_df['added_year'].between(2015, 2022))]
yearly = movies.groupby('added_year').size().reset_index(name='count')

years  = yearly['added_year'].astype(str).tolist()
counts = [int(v) for v in yearly['count']]
total  = sum(counts)

# Year-over-year deltas; first year is the absolute baseline
deltas  = [counts[0]] + [counts[i] - counts[i-1] for i in range(1, len(counts))]
measures = ['absolute'] + ['relative'] * (len(counts) - 1) + ['absolute']
x_vals  = years + ['Total\n2015–2022']
y_vals  = deltas + [total]

# Text: show sign on delta bars, plain number on baseline and total
text_labels = (
    [str(counts[0])]
    + [(f'+{d}' if d > 0 else str(d)) for d in deltas[1:]]
    + [str(total)]
)

# Year with the largest positive jump vs prior year
max_jump      = max(deltas[1:])
max_jump_idx  = deltas.index(max_jump)          # index within deltas list
max_jump_year = years[max_jump_idx]
# Running total at the TOP of that bar
running = [counts[0]] + [sum(counts[:i+1]) for i in range(1, len(counts))]
bar_top = running[max_jump_idx]

fig = go.Figure(go.Waterfall(
    orientation='v',
    measure=measures,
    x=x_vals,
    y=y_vals,
    text=text_labels,
    textposition='outside',
    increasing={'marker': {'color': '#2ca02c'}},   # green
    decreasing={'marker': {'color': '#d62728'}},   # red
    totals  ={'marker': {'color': 'steelblue'}},
    connector={'line': {'color': 'rgba(80,80,80,0.4)', 'dash': 'dot', 'width': 1}},
))

fig.add_annotation(
    x=max_jump_year,
    y=bar_top,
    text=f'Biggest jump: +{max_jump} titles',
    showarrow=True, arrowhead=2, yshift=18,
    font={'color': '#2ca02c', 'size': 11}
)

fig.update_layout(
    title='Netflix Grew Unevenly: 2017, 2020 & 2022 Saw Cuts — 659 Movies Added Overall (2015–2022)',
    xaxis_title='Year',
    yaxis_title='Titles Added vs Previous Year',
    showlegend=False,
    height=520,
)
fig.show()
